# 01 · Carga de datos y variable objetivo

**Objetivo:** cargar los microdatos de la Encuesta de Estructura Salarial (INE) de 2018 y 2022, construir la variable objetivo (salario bruto anual) y comprobar que coincide con las cifras oficiales publicadas por el INE.

Fuente: INE, www.ine.es.

### Preparación del entorno

Funciona tanto en local (abriendo el notebook dentro del repositorio clonado) como en **Google Colab**: si se abre en Colab sin el repositorio, lo descarga de GitHub automáticamente y pide subir los datos del INE.

In [ ]:
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/alfredobrocalserrano/tfm-salarios-ees"
EN_COLAB = "google.colab" in sys.modules


def buscar_raiz():
    """Devuelve la carpeta del proyecto (la que contiene src/) o None si no está."""
    for carpeta in [Path.cwd(), *Path.cwd().parents]:
        if (carpeta / "src").is_dir():
            return carpeta
    return None


RAIZ = buscar_raiz()
if RAIZ is None:
    if EN_COLAB:
        RAIZ = Path("/content/tfm-salarios-ees")
        if not RAIZ.exists():
            print("Descargando el repositorio del proyecto...")
            subprocess.run(["git", "clone", "-q", REPO_URL, str(RAIZ)], check=True)
    else:
        raise FileNotFoundError(
            "No se encuentra la carpeta src/ del proyecto. Clona el repositorio "
            f"({REPO_URL}) y abre el notebook desde dentro (ver README)."
        )

sys.path.insert(0, str(RAIZ))
print("Carpeta del proyecto:", RAIZ)

In [ ]:
# Solo en Colab: si faltan los microdatos del INE, se piden para subirlos
if EN_COLAB:
    from google.colab import files

    carpeta_raw = RAIZ / "data" / "raw"
    carpeta_raw.mkdir(parents=True, exist_ok=True)
    faltan = [a for a in (2018, 2022) if not (carpeta_raw / f"md_EES_{a}.parquet").exists()]
    if faltan:
        print("Sube los ficheros:", ", ".join(f"md_EES_{a}.parquet" for a in faltan))
        for nombre, contenido in files.upload().items():
            for anio in faltan:
                if str(anio) in nombre and nombre.endswith(".parquet"):
                    (carpeta_raw / f"md_EES_{anio}.parquet").write_bytes(contenido)
                    print("Guardado:", f"md_EES_{anio}.parquet")

In [ ]:
import pandas as pd

from src.etiquetas import aplicar_etiquetas, NOMBRES
from src.datos import (
    ANIOS, MEDIA_PUBLICADA_INE, DATA_PROCESSED,
    leer_ees, calcular_salario_anual, dias_anio_computables,
    media_ponderada, cargar_ees_unificada,
)

pd.set_option("display.max_columns", 60)

## 1. Lectura de los dos años

In [ ]:
ees18 = leer_ees(2018)
ees22 = leer_ees(2022)

print(f"2018: {ees18.shape[0]:,} trabajadores x {ees18.shape[1]} columnas")
print(f"2022: {ees22.shape[0]:,} trabajadores x {ees22.shape[1]} columnas")

## 2. Comprobación de que ambos años son compatibles

In [ ]:
print("Columnas solo en 2018:", sorted(set(ees18.columns) - set(ees22.columns)))
print("Columnas solo en 2022:", sorted(set(ees22.columns) - set(ees18.columns)))

tipos_distintos = [c for c in ees18.columns if c in ees22.columns and ees18[c].dtype != ees22[c].dtype]
print("Columnas con tipo distinto:", tipos_distintos)
print("Valores nulos 2018:", int(ees18.isna().sum().sum()), "| 2022:", int(ees22.isna().sum().sum()))

In [ ]:
# Las categorías de las variables clave son las mismas en los dos años
clave = ["NUTS1", "CNACE", "ESTRATO2", "SEXO", "TIPOPAIS", "CNO1", "ESTU", "ANOS2"]
pd.DataFrame({
    "categorias_2018": [ees18[c].nunique() for c in clave],
    "categorias_2022": [ees22[c].nunique() for c in clave],
    "mismos_codigos": [set(ees18[c]) == set(ees22[c]) for c in clave],
}, index=clave)

## 3. Variable objetivo: salario bruto anual

Siguiendo el documento del INE *"Obtención de los resultados publicados"*:

- **Salario bruto** = `RETRINOIN + RETRIIN + VESPNOIN + VESPIN` (retribuciones ordinarias y en especie, derivadas o no de incapacidad temporal).
- **Días computables** = `DRELABAM × 30,42 + DRELABAD − DSIESPA2 − DSIESPA4`.
- **Salario anual** = salario bruto × 365 / días computables (se eleva a año completo para quien no trabajó todo el año).
- Las medias se calculan **ponderando con `FACTOTAL`** (factor de elevación), para que representen al total de asalariados.

In [ ]:
for df in (ees18, ees22):
    df["dias_computables"] = dias_anio_computables(df)
    df["salario_anual"] = calcular_salario_anual(df)

ees22[["DRELABAM", "DRELABAD", "dias_computables", "RETRINOIN", "VESPNOIN", "salario_anual", "FACTOTAL"]].head()

## 4. Validación contra las cifras oficiales del INE

In [ ]:
validacion = pd.DataFrame({
    "media_calculada": [media_ponderada(df["salario_anual"], df["FACTOTAL"]) for df in (ees18, ees22)],
    "media_publicada_INE": [MEDIA_PUBLICADA_INE[a] for a in ANIOS],
}, index=ANIOS)
validacion["diferencia_%"] = 100 * (validacion["media_calculada"] / validacion["media_publicada_INE"] - 1)
validacion.round(2)

La media calculada coincide con la publicada por el INE con una diferencia inferior al 0,02 %, debida al redondeo. **La variable objetivo está bien construida.**

## 5. Unión de los dos años

In [ ]:
ees = cargar_ees_unificada()
print(f"Dataset unificado: {ees.shape[0]:,} filas x {ees.shape[1]} columnas")
ees.groupby("anio")["salario_anual"].describe().round(0)

## 6. Etiquetas legibles

Los códigos del INE (`CNO1 = "C0"`, `ESTU = "7"`...) se traducen a texto con el diseño de registro `dr_EES_2022.xlsx`. Se conservan las columnas originales y se añade una columna legible por variable (`ocupacion`, `estudios`, `region`...).

In [ ]:
ees = aplicar_etiquetas(ees)
ees[list(NOMBRES.values())].head()

In [ ]:
# Ejemplo: distribución de trabajadores por ocupación en 2022 (sin ponderar)
ees.loc[ees.anio == 2022, "ocupacion"].value_counts()

**Nota sobre el tamaño de empresa:** los códigos `0` y `4` de `ESTRATO2` no son un tamaño concreto, sino agrupaciones que el INE aplica para proteger el anonimato (≈2,4 % de los registros). Se tratarán en la fase de preparación.

In [ ]:
ees["tamano_empresa"].value_counts()

## 7. Guardado

In [ ]:
DATA_PROCESSED.mkdir(parents=True, exist_ok=True)
ees.to_parquet(DATA_PROCESSED / "ees_2018_2022.parquet", index=False)
print("Guardado en", DATA_PROCESSED / "ees_2018_2022.parquet")

## Pendiente

- Deflactar el salario de 2018 a euros de 2022 con el IPC.